# SnowHealth Demand Forecasting — Environment Setup

Creates the `SNOWHEALTH_ML_DB` database, a compute pool for ML workloads, generates 10,000 synthetic retail-pharmacy transactions, and loads them into Snowflake. Run all cells top to bottom, then proceed to `02_feature_engineering.ipynb`.

```
SNOWHEALTH_ML_DB
├── SNOWHEALTH_RAW          — Raw transaction data (created here)
├── SNOWHEALTH_FEATURES     — Feature Store (Notebook 02)
└── SNOWHEALTH_MODELS       — Model Registry & Monitoring (Notebooks 03–09)

SNOWHEALTH_COMPUTE_POOL     — SPCS compute for HPO tuning & model serving
```

**Prerequisites:** A role with `CREATE DATABASE` and `CREATE COMPUTE POOL` privileges and a running warehouse.

In [ ]:
import sys, os

# Resolve source/ (works from root or notebooks/ subdirectory)
for _p in [os.path.join(os.getcwd(), 'source'), os.path.join(os.getcwd(), '..', 'source')]:
    if os.path.isdir(_p): sys.path.insert(0, os.path.realpath(_p)); break

from snowflake.snowpark.context import get_active_session
import config as cfg

session = get_active_session()
print(f"Role:      {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Session ready.")

In [ ]:
%%sql -r db_result
CREATE DATABASE IF NOT EXISTS SNOWHEALTH_ML_DB
    COMMENT = 'SnowHealth Demand Forecasting ML Demo';

CREATE SCHEMA IF NOT EXISTS SNOWHEALTH_ML_DB.SNOWHEALTH_RAW
    COMMENT = 'Raw transaction data for demand forecasting';

CREATE SCHEMA IF NOT EXISTS SNOWHEALTH_ML_DB.SNOWHEALTH_FEATURES
    COMMENT = 'Engineered features and Feature Store objects';

CREATE SCHEMA IF NOT EXISTS SNOWHEALTH_ML_DB.SNOWHEALTH_MODELS
    COMMENT = 'Model registry, predictions, and monitoring';

In [ ]:
%%sql -r pool_result
-- Compute pool for ML Jobs (HPO tuning) and model serving (SPCS inference).
-- CPU_X64_S: 3 vCPUs, 13 GiB RAM per node — sufficient for XGBoost/LightGBM HPO trials.
CREATE COMPUTE POOL IF NOT EXISTS SNOWHEALTH_COMPUTE_POOL
    MIN_NODES = 1
    MAX_NODES = 5
    INSTANCE_FAMILY = CPU_X64_S
    AUTO_RESUME = TRUE
    AUTO_SUSPEND_SECS = 3600
    COMMENT = 'SnowHealth ML demo — HPO tuning, ML Jobs, and model serving';

In [ ]:
from synthetic_data import generate_transactions

df_raw = generate_transactions(n_rows=10000, seed=42)

# Set session context (CREATE DATABASE resets it)
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_RAW_SCHEMA}").collect()

# Load into Snowflake
df_load = df_raw.copy()
df_load['PROMOTION_FLAG'] = df_load['PROMOTION_FLAG'].astype(bool)
df_load['IS_WEEKEND']     = df_load['IS_WEEKEND'].astype(bool)
df_load['IS_HOLIDAY']     = df_load['IS_HOLIDAY'].astype(bool)

snow_df = session.create_dataframe(df_load)
snow_df.write.mode('overwrite').save_as_table(
    f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_RAW_SCHEMA}.SNOWHEALTH_DEMAND_TRANSACTIONS"
)

print(f"Generated and loaded {len(df_raw):,} rows into {cfg.SNOWHEALTH_TRANSACTIONS_TABLE}")

In [ ]:
%%sql -r data_summary
SELECT
    COUNT(*)                            AS TOTAL_ROWS,
    COUNT(DISTINCT STORE_ID)            AS NUM_STORES,
    COUNT(DISTINCT SKU_ID)              AS NUM_SKUS,
    COUNT(DISTINCT REGION)              AS NUM_REGIONS,
    COUNT(DISTINCT FULFILLMENT_CHANNEL) AS NUM_CHANNELS,
    COUNT(DISTINCT PRODUCT_CATEGORY)    AS NUM_CATEGORIES,
    MIN(TRANSACTION_DATE)               AS DATE_FROM,
    MAX(TRANSACTION_DATE)               AS DATE_TO
FROM SNOWHEALTH_ML_DB.SNOWHEALTH_RAW.SNOWHEALTH_DEMAND_TRANSACTIONS;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_raw['REGION'].value_counts().plot(kind='bar', ax=axes[0,0], color='#29B5E8')
axes[0,0].set_title('Transactions by Region')
axes[0,0].tick_params(axis='x', rotation=0)

df_raw['PRODUCT_CATEGORY'].value_counts().plot(kind='barh', ax=axes[0,1], color='#29B5E8')
axes[0,1].set_title('Transactions by Product Category')

df_raw['FULFILLMENT_CHANNEL'].value_counts().plot(kind='bar', ax=axes[1,0], color='#29B5E8')
axes[1,0].set_title('Transactions by Fulfillment Channel')
axes[1,0].tick_params(axis='x', rotation=15)

axes[1,1].hist(df_raw['UNITS_SOLD'], bins=40, color='#29B5E8', edgecolor='white')
axes[1,1].set_title('Distribution of Units Sold')
axes[1,1].set_xlabel('Units Sold')

plt.suptitle('SnowHealth Demand Data — Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
daily = df_raw.groupby('TRANSACTION_DATE')['UNITS_SOLD'].sum().reset_index()
daily['TRANSACTION_DATE'] = pd.to_datetime(daily['TRANSACTION_DATE'])
daily = daily.sort_values('TRANSACTION_DATE')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily['TRANSACTION_DATE'], daily['UNITS_SOLD'], color='#29B5E8', alpha=0.5, linewidth=0.8, label='Daily')
ax.plot(daily['TRANSACTION_DATE'], daily['UNITS_SOLD'].rolling(14).mean(), color='#333', linewidth=2, label='14-day avg')
ax.set_title('Daily Demand Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Units Sold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print('=' * 60)
print('  SETUP COMPLETE')
print('=' * 60)
print(f"  Database:  {cfg.SNOWHEALTH_DATABASE}")
print(f"  Schemas:   {cfg.SNOWHEALTH_RAW_SCHEMA}, {cfg.SNOWHEALTH_FEATURES_SCHEMA}, {cfg.SNOWHEALTH_MODELS_SCHEMA}")
print(f"  Table:     {cfg.SNOWHEALTH_TRANSACTIONS_TABLE} ({len(df_raw):,} rows)")
print(f"  Stores: {df_raw['STORE_ID'].nunique()} | SKUs: {df_raw['SKU_ID'].nunique()} | Regions: {df_raw['REGION'].nunique()} | Channels: {df_raw['FULFILLMENT_CHANNEL'].nunique()}")
print('=' * 60)
print('  Next → 02_feature_engineering.ipynb')